In [1]:
import numpy as np

import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
df=pd.read_csv(r"data\loandata.csv")


**Problem Statement**

- About Company

  - Dream Housing Finance company deals in all home loans. They have presence across all urban, semi urban and rural areas. Customer first apply for home loan after that company validates the customer eligibility for loan.

- Problem

  - Company wants to automate the loan eligibility process (real time) based on customer detail provided while filling online application form. These details are Gender, Marital Status, Education, Number of Dependents, Income, Loan Amount, Credit History and others. To automate this process, they have given a problem to identify the customers segments, those are eligible for loan amount so that they can specifically target these customers. Here they have provided a partial data set.

In [5]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001722,Male,Yes,0,Graduate,No,150,1800.0,135.0,360.0,1.0,Rural,N
1,LP002502,Female,Yes,2,Not Graduate,NaN,210,2917.0,98.0,360.0,1.0,Semiurban,Y
2,LP002949,Female,No,3+,Graduate,NaN,416,41667.0,350.0,180.0,NaN,Urban,N
3,LP002603,Female,No,0,Graduate,No,645,3683.0,113.0,480.0,1.0,Rural,Y
4,LP001644,NaN,Yes,0,Graduate,Yes,674,5296.0,168.0,360.0,1.0,Rural,Y


## Group by: split-apply-combine

By “group by” we are referring to a process involving one or more of the following steps:

- **Splitting** the data into groups based on some criteria.

- **Applying** a function to each group independently.

- **Combining** the results into a data structure.

Out of these, the split step is the most straightforward. In fact, in many situations we may wish to split the data set into groups and do something with those groups. In the apply step, we might wish to do one of the following:

**Aggregation** : compute a **summary statistic (or statistics)** for each group. Some examples:

- Compute group sums or means.

- Compute group sizes / counts.

**Transformation** : perform some group-specific computations and return a like-indexed object. Some examples:

- Standardize data (zscore) within a group.

- Filling NAs within groups with a value derived from each group.

**Filtration** : discard some groups, according to a group-wise computation that evaluates True or False. Some examples:

- Discard data that belongs to groups with only a few members.

- Filter out data based on the group sum or mean.

Some combination of the above: GroupBy will examine the results of the apply step and try to return a sensibly combined result if it doesn’t fit into either of the above two categories.





Syntax :



df.groupby(by = grouping_columns)[columns_to_show].function()

![image.png](attachment:image.png)

In [22]:
#Finding mean value of ApplicantIncome based on gender

df.groupby(by = "Gender")[["ApplicantIncome"]].mean()

,ApplicantIncome
Gender,
Female,4643.473214
Male,5446.460123


In [8]:
df.Education.unique()

array(['Graduate', 'Not Graduate'], dtype=object)

In [9]:
df.Education.value_counts()

Graduate        480
Not Graduate    134
Name: Education, dtype: int64

In [23]:
df.groupby(by = "Education")["ApplicantIncome"].sum()

Education
Graduate        2811568
Not Graduate     506156
Name: ApplicantIncome, dtype: int64

In [3]:
df.groupby(by = "Gender")[["ApplicantIncome","LoanAmount"]].mean()

,ApplicantIncome,LoanAmount
Gender,,
Female,4643.473214,126.697248
Male,5446.460123,149.265957


In [12]:
df.groupby(by = "Gender")[["ApplicantIncome","LoanAmount",'Credit_History']].mean()

,ApplicantIncome,LoanAmount,Credit_History
Gender,,,
Female,4643.473214,126.697248,0.831683
Male,5446.460123,149.265957,0.847007


In [13]:
type(df.groupby(by = "Gender")[["ApplicantIncome","LoanAmount",'Credit_History']].mean())

pandas.core.frame.DataFrame

### Using multiple columns for groupby

In [25]:
df.groupby(by = ["Married","Gender"])[["ApplicantIncome"]].sum()

ApplicantIncome
Married Gender                 
No      Female           360303
        Male             680699
Yes     Female           149719
        Male            1974046

In [15]:
type(df.groupby(by = ["Married","Gender"])[["ApplicantIncome"]].mean())

pandas.core.frame.DataFrame

In [16]:
df.groupby(by = ["Gender", 'Married'])[["ApplicantIncome"]].mean()

ApplicantIncome
Gender Married                 
Female No           4503.787500
       Yes          4829.645161
Male   No           5236.146154
       Yes          5529.540616

In [17]:
df.groupby(by = ["Gender", 'Married'])[["ApplicantIncome"]].mean().reset_index()

,Gender,Married,ApplicantIncome
0,Female,No,4503.787500
1,Female,Yes,4829.645161
2,Male,No,5236.146154
3,Male,Yes,5529.540616


In [18]:
df.groupby(by=['Gender','Married']).mean()

ApplicantIncome  CoapplicantIncome  LoanAmount  \
Gender Married                                                   
Female No           4503.787500        1020.012500  116.115385   
       Yes          4829.645161        1370.838710  153.322581   
Male   No           5236.146154        1529.430769  136.088000   
       Yes          5529.540616        1828.330308  154.011662   

                Loan_Amount_Term  Credit_History  
Gender Married                                    
Female No             355.012987        0.821918  
       Yes            349.161290        0.851852  
Male   No             348.562500        0.845528  
       Yes            335.931034        0.846626

In [19]:
df.groupby(by=['Gender','Married'])['ApplicantIncome'].max()    #max values in grouping gender and married

Gender  Married
Female  No         18165
        Yes        19484
Male    No         37719
        Yes        81000
Name: ApplicantIncome, dtype: int64

In [20]:
df['Gender'].isnull().sum()

13

### Aggregation Function on Groupby

![image.png](attachment:image.png)

In [26]:
df.groupby(['Gender','Married'])["Gender"].agg(['count'])     #gives no of counts yes|no

count
Gender Married       
Female No          80
       Yes         31
Male   No         130
       Yes        357

In [27]:
df.groupby(['Gender']).agg({'LoanAmount':np.mean,'ApplicantIncome': np.std, 'CoapplicantIncome':np.mean})                       #specified statistical values mean,min,max,std

,LoanAmount,ApplicantIncome,CoapplicantIncome
Gender,,,
Female,126.697248,3585.381488,1108.008929
Male,149.265957,6185.789262,1742.932352


In [28]:
df.groupby(['Gender'])['CoapplicantIncome'].mean()

Gender
Female    1108.008929
Male      1742.932352
Name: CoapplicantIncome, dtype: float64

**Note that df.groupby returns a DataFrameGroupBy object.**

In [35]:
df1=df.groupby(['Married',"Gender"])
df1

In [36]:
df1["ApplicantIncome"].mean()

Married  Gender
No       Female    4503.787500
         Male      5236.146154
Yes      Female    4829.645161
         Male      5529.540616
Name: ApplicantIncome, dtype: float64

In [37]:
df1["ApplicantIncome"].count()

Married  Gender
No       Female     80
         Male      130
Yes      Female     31
         Male      357
Name: ApplicantIncome, dtype: int64

## Crosstab
The pandas crosstab function builds a cross-tabulation table that **can show the frequency with which certain groups of data appear.**

Summary tables:

- Suppose we want to see how the observations in our sample are distributed in the context of two variables To do so, we can build a contingency table using the crosstab method:




pd.crosstab(column1, column2)

In [44]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001722,Male,Yes,0,Graduate,No,150,1800.0,135.0,360.0,1.0,Rural,N
1,LP002502,Female,Yes,2,Not Graduate,NaN,210,2917.0,98.0,360.0,1.0,Semiurban,Y
2,LP002949,Female,No,3+,Graduate,NaN,416,41667.0,350.0,180.0,NaN,Urban,N
3,LP002603,Female,No,0,Graduate,No,645,3683.0,113.0,480.0,1.0,Rural,Y
4,LP001644,NaN,Yes,0,Graduate,Yes,674,5296.0,168.0,360.0,1.0,Rural,Y


In [38]:
pd.crosstab(df["Gender"],df["Married"])

Married,No,Yes
Gender,,
Female,80,31
Male,130,357


### Question:  I want to compare Average Income  with respect to Gender and Married

**Using crosstab**

In [39]:
## USing Cross Tab


pd.crosstab(df["Gender"],df["Married"], values=df.ApplicantIncome, aggfunc='mean')

Married,No,Yes
Gender,,
Female,4503.787500,4829.645161
Male,5236.146154,5529.540616


**Using Groupby**

In [40]:
# Using Groupby:

df.groupby(by = ["Married","Gender"])["ApplicantIncome"].mean()

Married  Gender
No       Female    4503.787500
         Male      5236.146154
Yes      Female    4829.645161
         Male      5529.540616
Name: ApplicantIncome, dtype: float64

# Pivot Table

A pivot table is a data summarization tool that is used in the context of data processing.

Pivot tables are used to **summarize, sort, reorganize, group, count, total or average data stored in a database**.

It allows its users to transform columns into rows and rows into columns.

It allows grouping by any data field.

Pivot tables are the **perfect solution when you need to summarize and analyze large amounts of data.**



We will create a pivot table on **ApplicantIncome** and compare how marks in multiple subject are correlated to the salary.


In [48]:
df=pd.read_csv(r"data\loandata.csv")

In [49]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001722,Male,Yes,0,Graduate,No,150,1800.0,135.0,360.0,1.0,Rural,N
1,LP002502,Female,Yes,2,Not Graduate,NaN,210,2917.0,98.0,360.0,1.0,Semiurban,Y
2,LP002949,Female,No,3+,Graduate,NaN,416,41667.0,350.0,180.0,NaN,Urban,N
3,LP002603,Female,No,0,Graduate,No,645,3683.0,113.0,480.0,1.0,Rural,Y
4,LP001644,NaN,Yes,0,Graduate,Yes,674,5296.0,168.0,360.0,1.0,Rural,Y


In [41]:
#Pivot table on ApplicantIncome mean value based on gender

df.pivot_table(values="ApplicantIncome",index="Gender",aggfunc='mean')

,ApplicantIncome
Gender,
Female,4643.473214
Male,5446.460123


In [42]:
#Pivot table on ApplicantIncome mean value based on gender and married column

df.pivot_table(values="ApplicantIncome",index="Gender",columns = "Married",aggfunc='mean')

Married,No,Yes
Gender,,
Female,4503.787500,4829.645161
Male,5236.146154,5529.540616


In [43]:
#Pivot table on ApplicantIncome count value based on gender and married column

df.pivot_table(values="ApplicantIncome",index="Gender",columns = "Married",aggfunc='count')

Married,No,Yes
Gender,,
Female,80,31
Male,130,357


In [45]:
df.pivot_table(values='ApplicantIncome',index=['Gender','Dependents'],columns=['Married','Education'],aggfunc=['sum','count'])

sum                                        count  \
Married                  No                    Yes                    No   
Education          Graduate Not Graduate  Graduate Not Graduate Graduate   
Gender Dependents                                                          
Female 0           212327.0      58554.0   53031.0      18336.0     50.0   
       1            54022.0      13664.0   55584.0          NaN     10.0   
       2             7177.0          NaN   22558.0        210.0      2.0   
       3+            3499.0       1830.0       NaN          NaN      2.0   
Male   0           447545.0      97528.0  677415.0      98433.0     83.0   
       1            64774.0      13097.0  347336.0      49842.0      6.0   
       2            25561.0          NaN  354597.0      81264.0      6.0   
       3+           21667.0       4707.0  269897.0      51119.0      2.0   

                                                      
Married                             Yes               
Education         Not Graduate Graduate Not Graduate  
Gender Dependents                                     
Female 0                  10.0     15.0          5.0  
       1                   3.0      6.0          NaN  
       2                   NaN      4.0          1.0  
       3+                  1.0      NaN          NaN  
Male   0                  26.0    120.0         29.0  
       1                   4.0     58.0         14.0  
       2                   NaN     64.0         22.0  
       3+                  1.0     29.0         13.0